# 2장 Tokens and Embeddings — SageMaker AI Studio(JupyterLab)용

<!-- origin-notice -->
> **원본 노트북:** [chapter02/Chapter 2 - Tokens and Token Embeddings.ipynb](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter02/Chapter%202%20-%20Tokens%20and%20Token%20Embeddings.ipynb)  
> **원본 저장소:** [HandsOnLLM/Hands-On-Large-Language-Models](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models) (Apache License 2.0, © 2024 Jay Alammar & Maarten Grootendorst)  
> 이 노트북은 원본을 SageMaker AI Studio에서 실행되도록 수정하고 한국어 설명·보충 실험을 추가한 것입니다. 변경 내용은 노트북 안의 "책과 다를 수 있는 점"에 적혀 있습니다.

원본: HandsOnLLM/Hands-On-Large-Language-Models `chapter02` 노트북. 1장 노트북과 같은 space(`ml.g4dn.xlarge`, 스토리지 30GB 이상)에서 실행하면 Phi-3 가중치 캐시를 그대로 다시 씁니다.

| 절 | 하는 일 | 책 위치(PDF 쪽) |
|---|---|---|
| 1 | Phi-3에 넣기 전 텍스트가 토큰 ID로 바뀌고, 출력 ID가 다시 글자로 바뀌는 과정 보기 | 68–77 |
| 2 | 토크나이저 8종이 같은 문장을 어떻게 다르게 자르는지 비교 (+ 한국어 토큰 수 비교) | 78–93 |
| 3 | 언어 모델이 토큰마다 내는 **문맥 임베딩** 보기 (+ "bank" 실험: 층별 문맥화) | 94–99 |
| 4 | 문장 전체를 벡터 하나로 만드는 **텍스트 임베딩** (+ 문장 유사도) | 100–101 |
| 5 | 사전학습된 **단어 임베딩**(GloVe)으로 비슷한 단어 찾기 | 102–107 |
| 6 | word2vec을 **노래 재생목록**에 적용한 추천 시스템 | 108–113 |

`(+ …)` 표시는 책에 없는 보충 실험입니다. 셀은 위에서부터 차례로 실행하세요.

## 0. 환경 점검과 패키지 설치

GPU·디스크·버전을 확인하고, 이미지에 없는 패키지만 설치합니다. `gensim`은 4.4 이상이어야 numpy 2.x가 깔린 최신 이미지와 충돌하지 않습니다(책 고정 버전 4.3.2는 numpy<2 요구).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h ~ | tail -1
import importlib, subprocess, sys
need = {"sentence_transformers": "sentence-transformers", "gensim": "gensim>=4.4", "sentencepiece": "sentencepiece"}
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
import jinja2                       # 채팅 템플릿(apply_chat_template)은 jinja2 3.1 이상 필요
if tuple(map(int, jinja2.__version__.split(".")[:2])) < (3, 1):
    missing.append("jinja2>=3.1")
if missing:
    print("설치:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("추가 설치 불필요")
import torch, transformers
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(), "| transformers", transformers.__version__)

## 1. LLM 토큰화: 텍스트 → 토큰 ID → 텍스트 (PDF 68–77)

LLM은 글자를 직접 보지 않습니다. **토크나이저**가 텍스트를 토큰으로 자르고 각 토큰을 어휘집(vocabulary)의 번호(ID)로 바꾼 뒤에야 모델에 들어갑니다. 모델이 내놓는 것도 ID이고, 이를 다시 글자로 바꾸는 것도 토크나이저입니다. 그래서 1장에서 모델과 토크나이저를 **함께** 불러왔습니다.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",        # transformers 5.x에서는 dtype="auto" (기존 이름은 경고만 나고 동작)
    trust_remote_code=False,   # 책 본문은 True. 최신 transformers에는 Phi-3가 내장되어 False가 안전
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

채팅 템플릿을 쓰지 않고 프롬프트 끝에 `<|assistant|>`를 직접 붙였습니다. 이 특수 토큰이 "여기서부터 답을 쓰라"는 신호입니다.

> **책과 다를 수 있는 점.** 책과 공식 노트북의 출력은 맨 앞에 문장 시작 토큰 `<s>`(ID 1)가 붙어 있습니다. 현재 Hugging Face에 올라온 Phi-3 토크나이저 설정으로는 `<s>`가 붙지 않아 ID가 `14350`(Write)부터 시작할 수 있습니다. 토크나이저도 버전에 따라 바뀌는 "모델의 일부"라는 사례입니다. 나머지 ID는 책과 같습니다.

In [ ]:
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened.<|assistant|>"

input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")
generation_output = model.generate(input_ids=input_ids, max_new_tokens=20)
print(tokenizer.decode(generation_output[0]))

**모델이 실제로 받은 입력**은 이 정수 배열입니다.

In [ ]:
print(input_ids)

ID 하나씩 다시 글자로 바꿔 봅니다. **관찰 포인트:** `apolog`+`izing`, `trag`+`ic`처럼 한 단어가 여러 조각(subword)으로 잘립니다. 어휘집에 없는 단어도 조각을 조합해 표현할 수 있게 하려는 설계입니다.

In [ ]:
for id in input_ids[0]:
    print(tokenizer.decode(id))

출력에는 입력 ID 뒤에 **새로 생성된 ID**가 이어 붙어 있습니다(자기회귀 생성).

In [ ]:
generation_output

생성된 첫 단어 `Subject`도 사실은 토큰 두 개(3323 `Sub` + 622 `ject`)입니다.

In [ ]:
print(tokenizer.decode(3323))
print(tokenizer.decode(622))
print(tokenizer.decode([3323, 622]))
print(tokenizer.decode(29901))

**(+ 보충)** 책 91쪽 'Tokenizer Properties'의 세 요소, 즉 어휘 크기와 특수 토큰을 직접 확인합니다.

In [ ]:
print("어휘 크기(base):", tokenizer.vocab_size, "| 특수 토큰 포함:", len(tokenizer))
print("특수 토큰:", tokenizer.all_special_tokens)
print("채팅 템플릿 적용 예:\n", tokenizer.apply_chat_template(
    [{"role": "user", "content": "Hi"}], tokenize=False, add_generation_prompt=True))

다음 절부터는 Phi-3가 필요 없으므로 GPU 메모리를 비웁니다.

In [ ]:
import gc
del model, generation_output, input_ids
gc.collect(); torch.cuda.empty_cache()
print(f"GPU 사용량: {torch.cuda.memory_allocated()/1e9:.2f} GB")

## 2. 학습된 토크나이저 비교 (PDF 78–93)

같은 텍스트를 여러 토크나이저에 넣어 색으로 구분해 출력합니다. 토크나이저 파일만 내려받고 모델 가중치는 받지 않으므로 빠릅니다.

테스트 문장에는 대소문자, 이모지·한자, 파이썬 코드와 공백(탭), 숫자 계산이 섞여 있습니다. 토크나이저가 무엇을 위해 설계되었는지 드러나는 지점들입니다.

In [ ]:
from transformers import AutoTokenizer

colors_list = ['102;194;165', '252;141;98', '141;160;203',
               '231;138;195', '166;216;84', '255;217;47']

def show_tokens(sentence, tokenizer_name):
    try:
        tok = AutoTokenizer.from_pretrained(tokenizer_name)
    except Exception as e:
        print(f"[건너뜀] {tokenizer_name}: {type(e).__name__} — {str(e)[:120]}")
        return
    token_ids = tok(sentence).input_ids
    print(f"--- {tokenizer_name} | 어휘 {len(tok):,} | 토큰 {len(token_ids)}개")
    for idx, t in enumerate(token_ids):
        print(f'\x1b[0;30;48;2;{colors_list[idx % len(colors_list)]}m' + tok.decode(t) + '\x1b[0m', end=' ')
    print()

text = """
English and CAPITALIZATION
🎵 鸟
show_tokens False None elif == >= else: two tabs:"    " Three tabs: "       "
12.0*50=600
"""

**WordPiece (BERT, 2018).** uncased는 모두 소문자로 바꾸고 줄바꿈을 버립니다. 이모지·한자는 `[UNK]`(모르는 토큰)가 됩니다. `##`은 앞 토큰에 붙는 조각이라는 표시입니다. `[CLS]`/`[SEP]`는 1장 그림 1-21의 분류용 토큰입니다.

In [ ]:
show_tokens(text, "bert-base-uncased")
show_tokens(text, "bert-base-cased")

**BPE (GPT-2, 2019).** 줄바꿈과 공백을 보존하고, `[UNK]` 대신 이모지·한자를 바이트 조각으로 쪼갭니다(출력의 `�`). 공백 4개가 토큰 4개가 되는 등 코드에는 비효율적입니다.

In [ ]:
show_tokens(text, "gpt2")

**SentencePiece (Flan-T5, 2022).** 줄바꿈·공백 정보를 버리고 이모지·한자는 `<unk>`가 됩니다. 코드를 다루기 어려운 토크나이저입니다.

In [ ]:
show_tokens(text, "google/flan-t5-small")

**BPE, 어휘 약 10만 (GPT-4, 2023).** 공백 여러 개를 토큰 하나로, `elif`를 토큰 하나로 처리합니다. 코드에 맞춰 설계된 흔적입니다. 공식 구현은 `tiktoken`이고, 여기서는 Hugging Face에 올라온 동일 토크나이저를 씁니다.

In [ ]:
show_tokens(text, "Xenova/gpt-4")

**코드·과학 특화.** StarCoder2(2024)는 숫자를 **한 자리씩** 자릅니다(`600` → `6` `0` `0`). 자릿수를 일정하게 다뤄 산술에 유리하게 하려는 선택입니다. Galactica는 과학 문헌용으로 `[START_REF]` 같은 특수 토큰을 가집니다.

공식 노트북 주석에는 StarCoder2가 접근 승인(gated)이 필요하다고 되어 있습니다(2026-11 확인 시점에는 토크나이저가 승인 없이 내려받아졌습니다). 막히면 `[건너뜀]`으로 표시되니 `from huggingface_hub import login; login()`으로 토큰을 입력한 뒤 다시 실행하세요.

In [ ]:
show_tokens(text, "bigcode/starcoder2-15b")
show_tokens(text, "facebook/galactica-1.3b")

**Phi-3 (= Llama 2 토크나이저, BPE, 어휘 32,000).** 1절에서 쓴 바로 그 토크나이저입니다. 채팅용 특수 토큰(`<|user|>`, `<|assistant|>` 등)이 추가되어 있습니다.

In [ ]:
show_tokens(text, "microsoft/Phi-3-mini-4k-instruct")

### (+ 보충) 한국어 문장은 몇 토큰이 될까

같은 뜻이라도 토크나이저가 한국어를 얼마나 학습했는지에 따라 토큰 수가 크게 달라집니다. 토큰 수는 곧 **컨텍스트 창 소모량, API 비용, 생성 시간**입니다. 표에서 `토큰/글자`가 1에 가깝거나 넘으면 거의 글자(또는 바이트) 단위로 쪼개고 있다는 뜻입니다.

In [ ]:
import pandas as pd
samples = {
    "en": "The autonomous vehicle detected a pedestrian at the intersection.",
    "ko": "자율주행 차량이 교차로에서 보행자를 감지했다.",
}
names = ["bert-base-cased", "gpt2", "google/flan-t5-small", "Xenova/gpt-4", "microsoft/Phi-3-mini-4k-instruct"]
rows = []
for n in names:
    tok = AutoTokenizer.from_pretrained(n)
    r = {"tokenizer": n}
    for lang, s in samples.items():
        ids = tok(s, add_special_tokens=False).input_ids
        r[f"{lang}_tokens"] = len(ids)
        r[f"{lang}_토큰/글자"] = round(len(ids) / len(s), 2)
        r[f"{lang}_UNK"] = sum(i == tok.unk_token_id for i in ids) if tok.unk_token_id is not None else 0
    rows.append(r)
pd.DataFrame(rows).set_index("tokenizer")

## 3. 언어 모델의 문맥 임베딩 (PDF 94–99)

토큰 ID는 번호일 뿐이라, 모델은 먼저 ID마다 벡터(**토큰 임베딩**)를 꺼냅니다. 이 벡터가 Transformer 층을 지나면서 문장의 다른 토큰 정보와 섞여 **문맥 임베딩(contextualized embedding)**이 됩니다. 1장에서 다룬 "bank 문제"를 푸는 것이 이 단계입니다.

책 코드는 DeBERTa v3 xsmall의 출력을 봅니다. 결과 모양 `[1, 4, 384]`는 `[배치 1개, 토큰 4개, 벡터 차원 384]`입니다.

In [ ]:
from transformers import AutoModel, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-base")
model = AutoModel.from_pretrained("microsoft/deberta-v3-xsmall")

tokens = tokenizer('Hello world', return_tensors='pt')
output = model(**tokens)[0]
output.shape

In [ ]:
for token in tokens['input_ids'][0]:
    print(tokenizer.decode(token))

In [ ]:
output

> **책 코드 주의.** 위 셀은 책(97쪽)을 그대로 옮긴 것인데, 토크나이저(`deberta-base`, BPE 약 5만)와 모델(`deberta-v3-xsmall`, SentencePiece 약 12.8만)의 어휘집이 서로 다릅니다. 실행은 되고 모양도 맞지만, 모델은 ID 번호를 자기 어휘집 기준으로 해석하므로 벡터의 의미는 정확하지 않습니다. 실제로 쓸 때는 **모델과 같은 이름의 토크나이저**를 짝지어야 합니다. 아래 셀이 올바른 짝입니다.

In [ ]:
tok_v3 = AutoTokenizer.from_pretrained("microsoft/deberta-v3-xsmall")
t = tok_v3('Hello world', return_tensors='pt')
print([tok_v3.decode(i) for i in t['input_ids'][0]], model(**t)[0].shape)

### (+ 보충) "bank" 실험: 같은 단어, 다른 벡터

1장 Q&A에서 다룬 두 가지를 숫자로 확인합니다. BERT로 세 문장 속 `bank`의 벡터를 뽑아 코사인 유사도(1에 가까울수록 같은 방향)를 비교합니다.

- **토큰 임베딩(조회표):** ID로 표에서 꺼낸 벡터라 세 `bank`가 완전히 같습니다(유사도 1.0). word2vec과 같은 상태입니다.
- **0층:** BERT는 여기에 **위치 임베딩**을 더한 뒤 정규화합니다. 그래서 문장 속 위치가 다른 `bank`끼리는 이미 1.0보다 조금 낮습니다. money 문장과 river 문장은 `bank`가 둘 다 7번째 토큰이라 0층에서도 1.0이고, loan 문장만 2번째라 낮게 나옵니다. 1장 Q&A에서 "셀프 어텐션은 순서를 몰라서 위치를 따로 더해 준다"고 한 바로 그 장치입니다.
- **1층부터:** 셀프 어텐션이 주변 단어 정보를 섞어, 같은 뜻(돈-은행 / 대출-은행)끼리는 가깝고 다른 뜻(강둑)과는 멀어지는 경향이 보입니다.

In [ ]:
import torch.nn.functional as F
bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")
bert = AutoModel.from_pretrained("bert-base-uncased", output_hidden_states=True).eval()

sents = {"money": "I deposited my money at the bank.",
         "loan":  "The bank approved my loan yesterday.",
         "river": "We had a picnic on the bank of the river."}
bank_id = bert_tok.convert_tokens_to_ids("bank")
vecs = {}
with torch.no_grad():
    for k, s in sents.items():
        enc = bert_tok(s, return_tensors="pt")
        pos = (enc.input_ids[0] == bank_id).nonzero()[0].item()
        static = bert.embeddings.word_embeddings.weight[bank_id]   # 조회표의 bank 벡터
        hs = bert(**enc).hidden_states          # 13개: 0층(토큰+위치 임베딩) + 인코더 12층
        vecs[k] = torch.stack([static] + [h[0, pos] for h in hs])

cos = lambda a, b: F.cosine_similarity(a, b, dim=-1)
df = pd.DataFrame({
    "money-loan (same sense)":  cos(vecs["money"], vecs["loan"]).numpy(),
    "money-river (diff sense)": cos(vecs["money"], vecs["river"]).numpy(),
}, index=["lookup"] + [str(i) for i in range(13)]).round(3)
df.index.name = "layer"
df

In [ ]:
# 그래프 글자는 영어로 둡니다(Studio 기본 이미지에 한글 폰트가 없을 수 있음)
ax = df.plot(marker="o", figsize=(7, 3.5), title="cosine similarity of 'bank' vectors by layer")
ax.set_xlabel("layer (lookup = embedding table, 0 = token+position)"); ax.set_ylabel("cosine"); ax.grid(alpha=.3)

**읽는 법.** `lookup`은 둘 다 1.0입니다. 그 뒤 주황 선(다른 뜻)이 파란 선(같은 뜻)보다 아래로 벌어지면 문맥화가 일어난 것입니다. BERT 벡터는 전반적으로 서로 비슷한 방향을 향하는 성질이 있어(이방성, anisotropy) 절댓값보다 **두 선의 차이**를 보세요. 맨 위층에서 차이가 오히려 줄어들 수 있는데, 1장 Q&A에서 말한 "최상층은 사전학습 과제에 특화" 현상과 이어집니다.

In [ ]:
del model, bert; gc.collect()

## 4. 텍스트 임베딩: 문장·문서를 벡터 하나로 (PDF 100–101)

3절은 **토큰마다** 벡터가 하나씩 나왔습니다. 검색·분류·군집화에는 문장 전체를 대표하는 **벡터 하나**가 필요합니다. 가장 단순한 방법은 토큰 벡터들의 평균이고, `sentence-transformers` 모델은 이 용도로 따로 학습되어 있습니다(10장에서 직접 학습합니다).

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
vector = model.encode("Best movie ever!")
vector.shape

**(+ 보충)** 벡터 하나로 줄여도 의미가 남아 있는지 확인합니다. 단어가 거의 겹치지 않아도 뜻이 비슷한 문장끼리 유사도가 높게 나와야 합니다. 8장 시맨틱 검색(RAG)의 기반이 되는 성질입니다.

In [ ]:
from sentence_transformers import util
s = ["Best movie ever!",
     "I absolutely loved this film.",
     "The car stopped at the red light.",
     "The vehicle braked at the traffic signal."]
emb = model.encode(s, convert_to_tensor=True)
pd.DataFrame(util.cos_sim(emb, emb).cpu().numpy(), index=s, columns=[x[:18] for x in s]).round(2)

In [ ]:
del model; gc.collect(); torch.cuda.empty_cache()

## 5. LLM 이전의 단어 임베딩: GloVe (PDF 102–107)

word2vec·GloVe는 단어마다 **고정된** 벡터 하나를 줍니다(문맥 없음 — 3절 bank 실험의 0층과 같은 성질). 가볍고 빨라서 지금도 쓰입니다. 위키백과로 학습한 50차원 GloVe(66MB)를 받아 `king`과 가까운 단어를 찾습니다.

In [ ]:
import gensim.downloader as api
model = api.load("glove-wiki-gigaword-50")
model.most_similar([model['king']], topn=11)

**(+ 보충)** 책 1장 그림 1-8의 "벡터 연산으로 의미 관계 표현"을 확인합니다. `king - man + woman ≈ ?`

In [ ]:
model.most_similar(positive=['king', 'woman'], negative=['man'], topn=5)

## 6. 임베딩으로 노래 추천하기 (PDF 108–113)

word2vec의 핵심은 "**같이 등장하는 것끼리 가까운 벡터**"(skip-gram + negative sampling)입니다. 이 아이디어는 단어에만 쓰이지 않습니다. 재생목록을 문장, 노래를 단어로 보고 학습하면, 같은 재생목록에 자주 함께 들어가는 노래끼리 가까워집니다. 텍스트 없이 **노래 ID만으로** 추천이 됩니다.

데이터: 미국 라디오 방송국 재생목록(Cornell 공개 데이터). 인터넷에서 직접 받습니다.

In [ ]:
import pandas as pd
from urllib import request

data = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/train.txt')
lines = data.read().decode("utf-8").split('\n')[2:]           # 앞 두 줄은 메타데이터
playlists = [s.rstrip().split() for s in lines if len(s.split()) > 1]

songs_file = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/song_hash.txt')
songs_file = songs_file.read().decode("utf-8").split('\n')
songs = [s.rstrip().split('\t') for s in songs_file]
songs_df = pd.DataFrame(data=songs, columns=['id', 'title', 'artist']).set_index('id')
print(f"재생목록 {len(playlists):,}개, 노래 {len(songs_df):,}곡")

재생목록 하나 = 노래 ID의 나열. word2vec 입장에서는 "문장"입니다.

In [ ]:
print('Playlist #1:\n ', playlists[0], '\n')
print('Playlist #2:\n ', playlists[1])

학습 파라미터: `vector_size=32` 노래당 32차원 벡터, `window=20` 앞뒤 20곡까지를 "이웃"으로 봄, `negative=50` 이웃이 아닌 노래 50개를 뽑아 대조(negative sampling). CPU로 1~2분 걸립니다.

In [ ]:
from gensim.models import Word2Vec
model = Word2Vec(playlists, vector_size=32, window=20, negative=50, min_count=1, workers=4)

In [ ]:
song_id = 2172
model.wv.most_similar(positive=str(song_id))

In [ ]:
print(songs_df.iloc[2172])

ID만 보면 알 수 없으니 제목·가수로 바꿔 봅니다. 2172번(메탈)과 842번(힙합)의 추천이 장르별로 모이는지 확인하세요.

In [ ]:
import numpy as np
def print_recommendations(song_id):
    similar_songs = np.array(model.wv.most_similar(positive=str(song_id), topn=5))[:, 0]
    return songs_df.iloc[similar_songs]

print_recommendations(2172)

In [ ]:
print_recommendations(842)

## 7. 정리

| 개념 | 단위 | 문맥 반영 | 이 노트북 |
|---|---|---|---|
| 토큰 ID | 토큰 | — (번호일 뿐) | 1, 2절 |
| 정적 단어 임베딩 (word2vec·GloVe) | 단어 | 없음 | 5, 6절 / 3절 0층 |
| 문맥 임베딩 (BERT·DeBERTa 출력) | 토큰 | 있음 | 3절 |
| 텍스트 임베딩 (sentence-transformers) | 문장·문서 | 있음 | 4절 |

**자율주행·Agent로 이어지는 점(책 밖).** VLA·VLM은 카메라 이미지를 패치 단위 **비전 토큰**으로, 궤적을 **행동 토큰**으로 바꿔 같은 Transformer에 넣습니다. 이 노트북에서 본 "무엇을 토큰 하나로 볼 것인가(토크나이저 설계)"와 "같이 등장하는 것끼리 가깝게(임베딩 학습)"가 그대로 적용됩니다. 6절에서 노래 ID를 단어처럼 다룬 것이 그 축소판입니다.

**끝나면** Studio에서 space를 **Stop** 하세요. EBS 볼륨(모델 캐시)은 남아 다음 장에서 재사용됩니다.